# 2. Tumor tissue segmentation with a configurable 2D / 3D U-Net

Segments the four tumor tissues of the BraTS-PEDs labels:

| label | tissue |
|---|---|
| 1 | ET – enhancing tumor |
| 2 | NET – non-enhancing tumor |
| 3 | CC – cystic component |
| 4 | ED – peritumoral edema |

Evaluation also reports the tumor core (TC = 1+2+3) and whole tumor (WT = 1+2+3+4). Label meanings follow the BraTS-PEDs convention.

The labels are strongly imbalanced: NET is present in 256 of 257 Training subjects and makes up ~74 % of tumor voxels, while CC and ED are present in only 83 and 52 subjects. Sampling therefore oversamples rare tissues, and Dice is reported both over all subjects and over subjects where the structure is present.

Everything is driven by one configuration (`src/brats_peds_seg/config.py`, overridden by `configs/*.yaml`). This notebook walks through one experiment; `scripts/train_cv.py` runs the same thing from a terminal. Cells have not been executed in the published version.

In [ ]:
import os
from pathlib import Path

# Run from the repository root so the relative paths in the configuration resolve.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print(f"Working directory: {Path.cwd()}")

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from brats_peds_seg.config import load_config, resolve_config
from brats_peds_seg.folds import fold_summary, load_or_create_folds
from brats_peds_seg.inference import predict_folder
from brats_peds_seg.model import check_model
from brats_peds_seg.reporting import compare_runs, evaluate_folder, plot_prediction, plot_run, run_tables
from brats_peds_seg.metrics import summarize_metrics
from brats_peds_seg.training import run_cross_validation

## 2.1 Configuration

Options cover the input (histogram-matched or raw, sequences, normalization), 2D / 2.5D / 3D, the architecture (depth, width, residual blocks, SE / CBAM attention, attention gates on skips, conv / transformer / ASPP bottleneck, deep supervision), losses, optimisation, augmentation and inference. Set `smoke_test=True` for a two-minute end-to-end check.

In [ ]:
CONFIG = load_config("configs/3d_histmatched.yaml", overrides={
    # "smoke_test": True,
    # "epochs": 100,
})
cfg = resolve_config(CONFIG)
print(f"Experiment: {cfg['experiment_name']}")
print({k: cfg[k] for k in ["data_version", "dim", "modalities", "normalization", "bottleneck", "skip_attention",
                           "block_attention", "deep_supervision", "loss_region", "loss_pixel", "epochs"]})

## 2.2 Cross-validation split

Training uses only the 257 labelled Training subjects. Each subject has one multi-sequence scan, so a subject-level 5-fold split has no leakage between folds. Folds are stratified by data source (CBTN / DIPGr / DFCI) and by the presence of the rare labels CC and ED, and are saved once so that every experiment uses the same split. The BraTS-PEDs Validation subjects are kept as an external test set; their labels are not yet available.

In [ ]:
folds = load_or_create_folds(cfg)
display(fold_summary(cfg))

## 2.3 Model

The U-Net is built from the configuration. This check runs one training step at the patch size and reports parameters and peak GPU memory. Half-precision 3D convolutions use channels-last memory layout, because the default layout made cuDNN request a ~7 GB workspace for a single convolution.

In [ ]:
check_model(cfg)

## 2.4 Training

Each fold trains on the other four folds with random patches (3D) or slices (2D), validates every `val_every` epochs on its own fold, keeps the best checkpoint by `selection_metric`, and finally writes out-of-fold NIfTI predictions and per-subject metrics. Interrupted runs resume from `last.pt`; finished folds are skipped. Approximate cost on a 16 GB laptop GPU: 3D ≈ 2.2 min/epoch, 2.5D ≈ 0.6 min/epoch (250 steps per epoch).

In [ ]:
RUN_DIR = run_cross_validation(CONFIG)

## 2.5 Cross-validation results

Every Training subject is scored once by the model that did not see it. Dice per structure is reported over all subjects (absent structures count as 1 when the prediction is also empty, 0 otherwise) and over subjects where the structure is present. HD95 uses the BraTS convention of 373.13 mm when exactly one of prediction and reference is empty.

Note: the checkpoint of each fold is selected on that fold's validation subjects, which are also the subjects it is scored on, so these out-of-fold numbers are slightly optimistic. The external Validation set gives an unbiased estimate once its labels are available.

In [ ]:
summary, per_fold = run_tables(RUN_DIR)
display(summary.round(3))
plot_run(RUN_DIR)
plt.show()

## 2.6 Robustness across folds

A single mean hides how much the result depends on which subjects were held out. The spread of per-fold means shows that variability; for rare structures (CC, ED) a fold may contain only ~10 subjects with the structure, so their per-fold Dice is noisier.

In [ ]:
display(per_fold.round(3))
display(per_fold.agg(["mean", "std", "min", "max"]).round(3))

## 2.7 Visual check of out-of-fold predictions (worst and median subject)

In [ ]:
oof = pd.read_csv(Path(RUN_DIR) / "cv_metrics.csv").sort_values("mean_label_dice")
for case in [oof["case_id"].iloc[0], oof["case_id"].iloc[len(oof) // 2]]:
    plot_prediction(RUN_DIR, case, cfg["aligned_root"])
    plt.show()

## 2.8 Comparing experiments

In [ ]:
display(compare_runs(cfg["runs_root"]).round(3))

## 2.9 Predicting the external test set

The 5 fold models are averaged (optionally with flip test-time augmentation). Inputs must be prepared like the training data; for the BraTS-PEDs Validation subjects this is the matching Validation folder of the aligned or histogram-matched dataset. Predictions are written as `{case}.nii.gz` in the original grid.

In [ ]:
input_root = Path(cfg["histmatched_root"] if cfg["data_version"] == "histmatched" else cfg["aligned_root"]) / "Validation"
predictions = predict_folder(RUN_DIR, input_root, Path(RUN_DIR) / "predictions_Validation")

## 2.10 Scoring the external test set (when labels are available)

`label_root` must contain `{case}/{case}-seg.nii.gz` in the aligned space; apply the saved rigid transform of realigned Validation subjects to their labels first.

In [ ]:
VALIDATION_LABEL_ROOT = Path(cfg["aligned_root"]) / "Validation"  # update when labels are available
if any(VALIDATION_LABEL_ROOT.glob("*/*-seg.nii.gz")):
    metrics = evaluate_folder(predictions, VALIDATION_LABEL_ROOT)
    display(summarize_metrics(metrics).round(3))
else:
    print("No Validation labels yet.")